In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Optional

import cooler
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter
try:
    from skimage.metrics import structural_similarity as skimage_ssim
except Exception:
    skimage_ssim = None
try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        return None
    class Markdown(str):
        pass

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

BASE = Path(os.environ.get("EMMA_BASE_DIR", "/mnt/disk1/duanran/emhic_baseline"))
CODE_DIR = BASE / "emhic_code"
OUT_ROOT = Path(os.environ.get("EMMA_TABLE_PIPELINE_OUT", str(CODE_DIR / "EMMA_for_git" / "computed_tables")))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BIN_LIMIT = int(os.environ.get("EMMA_BIN_LIMIT", "2000"))
WINDOW = int(os.environ.get("EMMA_ANALYSIS_WINDOW", "500"))
QUICK_RUN = bool(int(os.environ.get("EMMA_QUICK_RUN", "0")))

METHOD_LABELS = {
    "EMMA_baseLinear": "Linear",
    "LightGBM": "LightGBM",
    "HiCPlus": "HiCPlus",
    "DeepHiC": "DeepHiC",
    "HiCARN": "HiCARN",
    "HiCDiff": "HiCDiff",
    "EMMA": "EMMA",
}
METHOD_ORDER = ["EMMA_baseLinear", "LightGBM", "HiCPlus", "DeepHiC", "HiCARN", "HiCDiff", "EMMA"]

DATASETS = {
    "H1_ESC_10kb": {
        "title": "Bulk Hi-C H1 ESC, 10 kb",
        "mcool_uri": str(BASE / "data" / "H1_ESC_4DNFI82R42AD.mcool") + "::resolutions/10000",
        "chroms": ["chr2", "chr3", "chr4", "chr6", "chr7"],
        "mask_strategy": "emma_meta",
        "use_method_mask": False,
        "methods": {
            "EMMA_baseLinear": {"dir": BASE / "EMMA_base" / "results" / "mask_predictions", "prefix": "H1_ESC_EMMA_baseLinear_{chrom}_10kb_mask"},
            "LightGBM": {"dir": BASE / "lightGBM" / "results" / "mask_predictions", "prefix": "H1_ESC_LightGBM_{chrom}_10kb_mask"},
            "HiCPlus": {"dir": BASE / "hicplus" / "results" / "mask_predictions", "prefix": "H1_ESC_HiCplus_{chrom}_10kb_mask"},
            "DeepHiC": {"dir": BASE / "deephic" / "results" / "mask_predictions", "prefix": "H1_ESC_DeepHiC_{chrom}_10kb_mask"},
            "HiCARN": {"dir": BASE / "hicarn" / "results" / "mask_predictions", "prefix": "H1_ESC_HiCARN1_{chrom}_10kb_mask"},
            "HiCDiff": {"dir": BASE / "hicdiff" / "results" / "mask_predictions", "prefix": "H1_ESC_HiCDiff_{chrom}_10kb_mask"},
            "EMMA": {"dir": BASE / "EMMA" / "results" / "mask_predictions_0_2000", "prefix": "H1_ESC_EMMA_{chrom}_10kb_mask"},
        },
    },
    "HFFc6_10kb": {
        "title": "Bulk Hi-C HFFc6, 10 kb",
        "mcool_uri": str(BASE / "data" / "HFFc6_4DNFIAVXXO55.mcool") + "::resolutions/10000",
        "chroms": ["chr2", "chr3", "chr4", "chr6", "chr7"],
        "mask_strategy": "fixed_blocks",
        "use_method_mask": True,
        "methods": {
            "EMMA_baseLinear": {"dir": BASE / "EMMA_base" / "results_hffc6" / "mask_predictions", "prefix": "HFFc6_EMMA_baseLinear_{chrom}_10kb_mask"},
            "LightGBM": {"dir": BASE / "lightGBM" / "results_hffc6" / "mask_predictions", "prefix": "HFFc6_LightGBM_{chrom}_10kb_mask"},
            "HiCPlus": {"dir": BASE / "hicplus" / "results_hffc6" / "mask_predictions", "prefix": "HFFc6_HiCplus_{chrom}_10kb_mask"},
            "DeepHiC": {"dir": BASE / "deephic" / "results_hffc6" / "mask_predictions", "prefix": "HFFc6_DeepHiC_{chrom}_10kb_mask"},
            "HiCARN": {"dir": BASE / "hicarn" / "results_hffc6" / "mask_predictions", "prefix": "HFFc6_HiCARN1_{chrom}_10kb_mask"},
            "HiCDiff": {"dir": BASE / "hicdiff" / "results_hffc6" / "mask_predictions", "prefix": "HFFc6_HiCDiff_{chrom}_10kb_mask"},
            "EMMA": {"dir": BASE / "EMMA" / "results_hffc6" / "mask_predictions_0_2000", "prefix": "HFFc6_EMMA_{chrom}_10kb_mask"},
        },
    },
    "PoreC_50kb": {
        "title": "Pore-C GM12878, 50 kb",
        "mcool_uri": str(BASE / "data" / "GSM4490689_GM12878_DpnII_69689_GRCh38_bwa_0.7.17_sensitive_GIABhiconf_whatshap_0.19.c01520_default_cf00.mcool") + "::resolutions/50000",
        "chroms": ["chr2", "chr3", "chr4", "chr6", "chr7"],
        "mask_strategy": "fixed_blocks",
        "use_method_mask": True,
        "methods": {
            "EMMA_baseLinear": {"dir": BASE / "EMMA_base" / "results_porec50" / "mask_predictions", "prefix": "PoreC_GM12878_EMMA_baseLinear_{chrom}_50kb_mask"},
            "LightGBM": {"dir": BASE / "lightGBM" / "results_porec50" / "mask_predictions", "prefix": "PoreC_GM12878_LightGBM_{chrom}_50kb_mask"},
            "HiCPlus": {"dir": BASE / "hicplus" / "results_porec50" / "mask_predictions", "prefix": "PoreC_GM12878_HiCplus_{chrom}_50kb_mask"},
            "DeepHiC": {"dir": BASE / "deephic" / "results_porec50" / "mask_predictions", "prefix": "PoreC_GM12878_DeepHiC_{chrom}_50kb_mask"},
            "HiCARN": {"dir": BASE / "hicarn" / "results_porec50" / "mask_predictions", "prefix": "PoreC_GM12878_HiCARN1_{chrom}_50kb_mask"},
            "HiCDiff": {"dir": BASE / "hicdiff" / "results_porec50" / "mask_predictions", "prefix": "PoreC_GM12878_HiCDiff_{chrom}_50kb_mask"},
            "EMMA": {"dir": BASE / "EMMA" / "results_porec50" / "mask_predictions_0_2000", "prefix": "PoreC_GM12878_EMMA_{chrom}_50kb_mask"},
        },
    },
    "MERFISH_100kb": {
        "title": "MERFISH IMR90, 100 kb",
        "mcool_uri": str(BASE / "data" / "fish_data" / "IMR90_MERFISH_100kb.mcool") + "::resolutions/100000",
        "chroms": ["chr2", "chr3", "chr4", "chr6", "chr7"],
        "mask_strategy": "fixed_blocks",
        "use_method_mask": True,
        "methods": {
            "EMMA_baseLinear": {"dir": BASE / "EMMA_base" / "results_fish100" / "mask_predictions", "prefix": "IMR90_MERFISH_EMMA_baseLinear_{chrom}_100kb_mask"},
            "LightGBM": {"dir": BASE / "lightGBM" / "results_fish100" / "mask_predictions", "prefix": "IMR90_MERFISH_LightGBM_{chrom}_100kb_mask"},
            "HiCPlus": {"dir": BASE / "hicplus" / "results_fish100" / "mask_predictions", "prefix": "IMR90_MERFISH_HiCplus_{chrom}_100kb_mask"},
            "DeepHiC": {"dir": BASE / "deephic" / "results_fish100" / "mask_predictions", "prefix": "IMR90_MERFISH_DeepHiC_{chrom}_100kb_mask"},
            "HiCARN": {"dir": BASE / "hicarn" / "results_fish100" / "mask_predictions", "prefix": "IMR90_MERFISH_HiCARN1_{chrom}_100kb_mask"},
            "HiCDiff": {"dir": BASE / "hicdiff" / "results_fish100" / "mask_predictions", "prefix": "IMR90_MERFISH_HiCDiff_{chrom}_100kb_mask"},
            "EMMA": {"dir": BASE / "EMMA" / "results_fish100" / "mask_predictions_dynamic", "prefix": "IMR90_MERFISH_EMMA_{chrom}_100kb_mask"},
        },
    },
}

BLOCK_SIZES = [5, 10, 15, 20, 25, 30]
GAP = 300
START_BIN = 200
MASK_MAX_DIAG = 500


def method_paths(dataset, method, chrom):
    cfg = dataset["methods"][method]
    stem = cfg["dir"] / cfg["prefix"].format(chrom=chrom)
    return {"filled": Path(str(stem) + "_filled.npy"), "mask": Path(str(stem) + "_mask.npy"), "meta": Path(str(stem) + "_meta.json")}


def missing_paths(dataset, method, chrom):
    return [str(p) for p in method_paths(dataset, method, chrom).values() if not p.exists()]


def load_prediction(dataset, method, chrom, n_eval):
    paths = method_paths(dataset, method, chrom)
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError("; ".join(missing))
    filled = np.load(paths["filled"], mmap_mode="r")[:n_eval, :n_eval].astype(np.float32)
    mask = np.load(paths["mask"], mmap_mode="r")[:n_eval, :n_eval].astype(bool)
    with open(paths["meta"], "r", encoding="utf-8") as f:
        meta = json.load(f)
    return filled, mask, meta


def complete_chromosomes(dataset):
    complete = []
    missing_records = []
    for chrom in dataset["chroms"]:
        missing_for_chrom = []
        for method in METHOD_ORDER:
            miss = missing_paths(dataset, method, chrom)
            if miss:
                missing_for_chrom.append(method)
                missing_records.append({"dataset": dataset["title"], "method": method, "chrom": chrom, "missing": "; ".join(miss)})
        if not missing_for_chrom:
            complete.append(chrom)
    return (complete[:1] if QUICK_RUN and complete else complete), missing_records


def chrom_eval_bins(dataset, chrom):
    clr = cooler.Cooler(dataset["mcool_uri"])
    start, end = clr.extent(chrom)
    return min(BIN_LIMIT, int(end - start))


def load_truth_matrix(dataset, chrom, n_eval):
    clr = cooler.Cooler(dataset["mcool_uri"])
    start, _ = clr.extent(chrom)
    mat = clr.matrix(balance=False)[start:start + n_eval, start:start + n_eval].astype(np.float32)
    mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
    mat[mat < 0] = 0.0
    return mat


def fixed_block_mask(n):
    mask = np.zeros((n, n), dtype=bool)
    regions = []
    selected = []
    cursor = START_BIN
    for block_size in BLOCK_SIZES:
        start = cursor
        end = min(n, start + int(block_size))
        if start >= n:
            break
        selected.extend(range(start, end))
        regions.append((start, end, int(block_size)))
        cursor = end + GAP
    for b in selected:
        lo = max(0, b - MASK_MAX_DIAG)
        hi = min(n, b + MASK_MAX_DIAG + 1)
        mask[b, lo:hi] = True
        mask[lo:hi, b] = True
    return mask, regions


def fixed_window_bounds(center, n, size=WINDOW):
    size = min(int(size), int(n))
    start = int(center) - size // 2
    start = max(0, min(start, n - size))
    return start, start + size


def mask_boundary_windows(regions, n, size=WINDOW):
    windows = []
    seen = set()
    for item in regions:
        start, end = int(item[0]), int(item[1])
        block_size = int(item[2]) if len(item) > 2 else int(end - start)
        for side, center in [("upstream_boundary", start), ("downstream_boundary", end - 1)]:
            s, e = fixed_window_bounds(center, n, size)
            key = (side, s, e)
            if key in seen:
                continue
            seen.add(key)
            windows.append({"side": side, "region_start": start, "region_end": end, "block_size": block_size, "start": s, "end": e})
    return windows


def minmax_scale_with_reference(x, ref, mask=None):
    vals = ref[np.isfinite(ref)] if mask is None else ref[mask & np.isfinite(ref)]
    if len(vals) == 0:
        return np.zeros_like(x, dtype=np.float32), 0.0, 1.0
    lo = float(np.nanmin(vals))
    hi = float(np.nanmax(vals))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-8:
        hi = lo + 1.0
    out = np.clip((x.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return out.astype(np.float32), lo, hi


def pair_scale_window(pred, true):
    true_s, lo, hi = minmax_scale_with_reference(true, true)
    pred_s = np.clip((pred.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return pred_s.astype(np.float32), true_s.astype(np.float32)


def pair_scale_window_on_mask(pred, true, mask):
    mask = np.asarray(mask, dtype=bool)
    vals = true[mask & np.isfinite(true)]
    if len(vals) == 0:
        return np.zeros_like(pred, dtype=np.float32), np.zeros_like(true, dtype=np.float32)
    lo = float(np.nanmin(vals))
    hi = float(np.nanmax(vals))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-8:
        hi = lo + 1.0
    pred_s = np.zeros_like(pred, dtype=np.float32)
    true_s = np.zeros_like(true, dtype=np.float32)
    pred_s[mask] = np.clip((pred[mask].astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    true_s[mask] = np.clip((true[mask].astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return pred_s, true_s


def hicrep_scc(pred, true, max_diag: Optional[int] = None, smooth_size=3):
    pred = np.nan_to_num(pred.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    true = np.nan_to_num(true.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if smooth_size and smooth_size > 1:
        pred = uniform_filter(pred, size=smooth_size, mode="nearest")
        true = uniform_filter(true, size=smooth_size, mode="nearest")
    n = pred.shape[0]
    max_diag = n - 1 if max_diag is None else int(max_diag)
    corrs = []
    weights = []
    for k in range(1, min(max_diag, n - 1) + 1):
        x = np.diagonal(pred, offset=k)
        y = np.diagonal(true, offset=k)
        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() < 3:
            continue
        x = x[valid]
        y = y[valid]
        sx = float(np.std(x))
        sy = float(np.std(y))
        if sx < 1e-8 or sy < 1e-8:
            continue
        r = float(np.corrcoef(x, y)[0, 1])
        if np.isfinite(r):
            corrs.append(r)
            weights.append(len(x) * sx * sy)
    return float(np.average(corrs, weights=weights)) if weights and np.sum(weights) > 1e-8 else np.nan


def row_normalize(mat):
    mat = np.nan_to_num(mat.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    mat[mat < 0] = 0.0
    mat = 0.5 * (mat + mat.T)
    row_sum = mat.sum(axis=1, keepdims=True)
    return mat / (row_sum + 1e-8)


def genomedisco_score(pred, true, t_list=(1, 2, 3), restart=0.5):
    px = row_normalize(pred)
    py = row_normalize(true)
    eye = np.eye(px.shape[0], dtype=np.float64)
    sx = eye.copy()
    sy = eye.copy()
    scores = []
    for step in range(1, max(t_list) + 1):
        sx = restart * eye + (1.0 - restart) * sx.dot(px)
        sy = restart * eye + (1.0 - restart) * sy.dot(py)
        if step in t_list:
            diff = np.abs(sx - sy).sum()
            denom = np.abs(sx).sum() + np.abs(sy).sum() + 1e-8
            scores.append(1.0 - diff / denom)
    return float(np.mean(scores)) if scores else np.nan


def ssim_score(pred, true):
    pred_s, true_s = pair_scale_window(pred, true)
    if skimage_ssim is not None:
        return float(skimage_ssim(true_s, pred_s, data_range=1.0))
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    mx = float(np.mean(pred_s))
    my = float(np.mean(true_s))
    vx = float(np.var(pred_s))
    vy = float(np.var(true_s))
    cov = float(np.mean((pred_s - mx) * (true_s - my)))
    return float(((2 * mx * my + c1) * (2 * cov + c2)) / ((mx * mx + my * my + c1) * (vx + vy + c2)))


def compute_dataset(dataset_key):
    dataset = DATASETS[dataset_key]
    out_dir = OUT_ROOT / dataset_key
    out_dir.mkdir(parents=True, exist_ok=True)
    complete, missing = complete_chromosomes(dataset)
    pd.DataFrame(missing).to_csv(out_dir / "missing_outputs.csv", index=False)
    if not complete:
        raise RuntimeError(f"No complete chromosomes for {dataset_key}; see {out_dir / 'missing_outputs.csv'}")
    records = []
    window_records = []
    for chrom in complete:
        n_eval = chrom_eval_bins(dataset, chrom)
        truth = load_truth_matrix(dataset, chrom, n_eval)
        if dataset["mask_strategy"] == "emma_meta":
            _, common_mask, emma_meta = load_prediction(dataset, "EMMA", chrom, n_eval)
            common_mask = common_mask & np.isfinite(truth)
            regions = emma_meta["regions"]
        else:
            common_mask, regions = fixed_block_mask(n_eval)
            common_mask = common_mask & np.isfinite(truth)
        windows = mask_boundary_windows(regions, n_eval, WINDOW)
        truth_scaled_mask, mask_lo, mask_hi = minmax_scale_with_reference(truth, truth, mask=common_mask)
        for method in METHOD_ORDER:
            filled, method_mask, _ = load_prediction(dataset, method, chrom, n_eval)
            eval_mask = common_mask & np.isfinite(filled)
            if dataset["use_method_mask"]:
                eval_mask = eval_mask & method_mask
            pred_scaled = np.clip((filled.astype(np.float32) - mask_lo) / (mask_hi - mask_lo), 0.0, 1.0)
            y_pred = pred_scaled[eval_mask]
            y_true = truth_scaled_mask[eval_mask]
            mse = float(np.mean((y_pred - y_true) ** 2)) if len(y_true) else np.nan
            records.append({"dataset": dataset["title"], "method": method, "chrom": chrom, "mask_points": int(eval_mask.sum()), "mask_mse": mse})
            for window_id, w in enumerate(windows):
                s, e = w["start"], w["end"]
                pred_w_s, true_w_s = pair_scale_window_on_mask(filled[s:e, s:e], truth[s:e, s:e], eval_mask[s:e, s:e])
                window_records.append({
                    "dataset": dataset["title"],
                    "method": method,
                    "chrom": chrom,
                    "window_id": window_id,
                    "window_mask_points": int(eval_mask[s:e, s:e].sum()),
                    "hicrep": hicrep_scc(pred_w_s, true_w_s),
                    "genomedisco": genomedisco_score(pred_w_s, true_w_s),
                    "ssim": ssim_score(pred_w_s, true_w_s),
                })
    mask_df = pd.DataFrame(records)
    window_df = pd.DataFrame(window_records)
    mask_df.to_csv(out_dir / "mask_metrics_by_chrom.csv", index=False)
    window_df.to_csv(out_dir / "structure_metrics_by_window.csv", index=False)
    summary = summarize_dataset(mask_df, window_df)
    summary.to_csv(out_dir / "summary_mean_std.csv", index=False)
    return summary, mask_df, window_df


def summarize_dataset(mask_df, window_df):
    mse = mask_df.groupby(["dataset", "method"], as_index=False).agg(
        mse_mean=("mask_mse", "mean"), mse_std=("mask_mse", "std"), chromosomes=("chrom", "nunique")
    )
    metrics = window_df.groupby(["dataset", "method"], as_index=False).agg(
        hicrep_mean=("hicrep", "mean"), hicrep_std=("hicrep", "std"),
        genomedisco_mean=("genomedisco", "mean"), genomedisco_std=("genomedisco", "std"),
        ssim_mean=("ssim", "mean"), ssim_std=("ssim", "std"), windows=("window_id", "count")
    )
    summary = mse.merge(metrics, on=["dataset", "method"])
    summary["method_order"] = summary["method"].map({m: i for i, m in enumerate(METHOD_ORDER)})
    return summary.sort_values(["dataset", "method_order"]).reset_index(drop=True)


def fmt_pm(mean, std, percent=False, decimals=2):
    mean = mean * 100.0 if percent else mean
    std = std * 100.0 if percent and not pd.isna(std) else std
    return f"{mean:.{decimals}f}" if pd.isna(std) else f"{mean:.{decimals}f} ± {std:.{decimals}f}"


def format_baseline_table(summary):
    rows = []
    for _, row in summary.iterrows():
        rows.append({
            "Dataset": row["dataset"],
            "Method": METHOD_LABELS[row["method"]],
            "MSE ↓": fmt_pm(row["mse_mean"], row["mse_std"], decimals=6),
            "HiCRep (%) ↑": fmt_pm(row["hicrep_mean"], row["hicrep_std"], percent=True, decimals=2),
            "GDISCO (%) ↑": fmt_pm(row["genomedisco_mean"], row["genomedisco_std"], percent=True, decimals=2),
            "SSIM (%) ↑": fmt_pm(row["ssim_mean"], row["ssim_std"], percent=True, decimals=2),
            "Chromosomes": int(row["chromosomes"]),
            "Windows": int(row["windows"]),
        })
    return pd.DataFrame(rows)


def compute_ablation_table():
    result_dir = CODE_DIR / "emma_ablation_multiregion_500_quadmaskae_fullwindow_e8n2000"
    files = sorted(result_dir.glob("region_*_results.csv"))
    files = files[:2] if QUICK_RUN else files
    if not files:
        raise FileNotFoundError(f"No ablation region result CSV files found in {result_dir}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    model_order = ["EMMA", "w/o EMD decomposition", "w/o Masked Autoencoder correction", "w/o mode-weighted reconstruction", "w/o distance-diagonal normalization"]
    short = {
        "EMMA": "EMMA",
        "w/o EMD decomposition": "- EMD",
        "w/o Masked Autoencoder correction": "- MaskAE",
        "w/o mode-weighted reconstruction": "- MWR",
        "w/o distance-diagonal normalization": "- DDN",
    }
    rows = []
    for model in model_order:
        sub = df[df["Model"].eq(model)]
        row = {"Variant": short[model], "Regions": int(sub["Region"].nunique())}
        for col in ["Mask MSE ↓", "HiCRep (%) ↑", "GenomeDISCO (%) ↑", "SSIM (%) ↑", "DI Curve Range (%) ↑"]:
            vals = pd.to_numeric(sub[col], errors="coerce")
            row[col] = fmt_pm(vals.mean(), vals.std(ddof=1), decimals=6 if col == "Mask MSE ↓" else 2)
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(OUT_ROOT / "table4_ablation_recomputed.csv", index=False)
    return out


def display_by_dataset(table, title):
    display(Markdown(f"## {title}"))
    for dataset_name, sub in table.groupby("Dataset", sort=False):
        display(Markdown(f"### {dataset_name}"))
        display(sub.drop(columns=["Dataset"]).reset_index(drop=True))

In [2]:
summary_h1, detail_h1_mask, detail_h1_window = compute_dataset("H1_ESC_10kb")
summary_hffc6, detail_hffc6_mask, detail_hffc6_window = compute_dataset("HFFc6_10kb")
TABLE_II_SUMMARY = pd.concat([summary_h1, summary_hffc6], ignore_index=True)
TABLE_II = format_baseline_table(TABLE_II_SUMMARY)


In [3]:
display_by_dataset(TABLE_II, "Table II. Bulk Hi-C baseline comparison")

## Table II. Bulk Hi-C baseline comparison

### Bulk Hi-C H1 ESC, 10 kb

,Method,MSE ↓,HiCRep (%) ↑,GDISCO (%) ↑,SSIM (%) ↑,Chromosomes,Windows
0,Linear,0.458 ± 0.28,52.61 ± 4.3,74.35 ± 8.2,77.74 ± 7.2,5,60
1,LightGBM,0.351 ± 0.01,57.63 ± 1.9,74.06 ± 1.3,88.72 ± 0.7,5,60
2,HiCPlus,0.236 ± 0.09,73.35 ± 19.2,80.95 ± 5.1,93.31 ± 2.8,5,60
3,DeepHiC,0.237 ± 0.06,77.58 ± 16.5,84.03 ± 3.9,93.26 ± 2.6,5,60
4,HiCARN,0.238 ± 0.08,87.57 ± 18.4,75.11 ± 5.8,93.24 ± 2.8,5,60
5,HiCDiff,0.231 ± 0.09,80.64 ± 6.3,86.76 ± 1.1,93.54 ± 2.7,5,60
6,EMMA,0.146 ± 0.05,91.85 ± 4.1,89.24 ± 0.9,96.02 ± 5.5,5,60


### Bulk Hi-C HFFc6, 10 kb

,Method,MSE ↓,HiCRep (%) ↑,GDISCO (%) ↑,SSIM (%) ↑,Chromosomes,Windows
0,Linear,0.575 ± 0.24,52.49 ± 4.28,72.45 ± 2.0,77.22 ± 10.5,5,60
1,LightGBM,0.451 ± 0.01,58.52 ± 1.34,72.36 ± 1.7,88.80 ± 0.7,5,60
2,HiCPlus,0.257 ± 0.08,61.70 ± 19.1,79.62 ± 4.5,93.36 ± 3.3,5,60
3,DeepHiC,0.251 ± 0.09,65.31 ± 15.6,76.45 ± 3.7,93.24 ± 3.3,5,60
4,HiCARN,0.252 ± 0.08,84.55 ± 14.0,73.56 ± 5.4,93.23 ± 3.3,5,60
5,HiCDiff,0.245 ± 0.08,78.93 ± 6.8,84.79 ± 2.3,91.50 ± 3.2,5,60
6,EMMA,0.149 ± 0.05,92.41 ± 2.8,89.52 ± 2.4,95.97 ± 1.9,5,60


In [4]:
summary_porec, detail_porec_mask, detail_porec_window = compute_dataset("PoreC_50kb")
summary_merfish, detail_merfish_mask, detail_merfish_window = compute_dataset("MERFISH_100kb")
TABLE_III_SUMMARY = pd.concat([summary_porec, summary_merfish], ignore_index=True)
TABLE_III = format_baseline_table(TABLE_III_SUMMARY)
TABLE_III.to_csv(OUT_ROOT / "table3_porec_merfish_recomputed.csv", index=False)


In [5]:
display_by_dataset(TABLE_III, "Table III. Pore-C and MERFISH baseline comparison")

## Table III. Pore-C and MERFISH baseline comparison

### Pore-C GM12878, 50 kb

,Method,MSE ↓,HiCRep (%) ↑,GDISCO (%) ↑,SSIM (%) ↑,Chromosomes,Windows
0,Linear,0.454 ± 0.07,56.73 ± 7.1,71.74 ± 2.6,78.45 ± 12.3,5,60
1,LightGBM,0.363 ± 0.10,58.37 ± 4.8,71.49 ± 1.9,80.87 ± 9.3,5,60
2,HiCPlus,0.543 ± 0.02,67.22 ± 15.4,72.64 ± 1.7,87.10 ± 5.7,5,60
3,DeepHiC,0.605 ± 0.02,72.76 ± 10.0,72.54 ± 1.6,83.43 ± 5.3,5,60
4,HiCARN,0.597 ± 0.02,85.52 ± 14.8,70.26 ± 2.9,85.66 ± 5.6,5,60
5,HiCDiff,0.181 ± 0.04,85.31 ± 6.9,71.85 ± 1.5,87.95 ± 11.9,5,60
6,EMMA,0.178 ± 0.01,93.69 ± 5.8,73.79 ± 1.7,90.10 ± 2.5,5,60


### MERFISH IMR90, 100 kb

,Method,MSE ↓,HiCRep (%) ↑,GDISCO (%) ↑,SSIM (%) ↑,Chromosomes,Windows
0,Linear,0.235 ± 0.14,46.17 ± 21.5,67.48 ± 1.1,41.97 ± 22.0,5,60
1,LightGBM,0.220 ± 0.02,42.69 ± 16.6,67.50 ± 1.2,45.10 ± 18.8,5,60
2,HiCPlus,0.257 ± 0.07,59.81 ± 16.4,77.50 ± 1.2,65.98 ± 14.7,5,60
3,DeepHiC,0.378 ± 0.06,59.62 ± 19.8,81.65 ± 6.6,60.98 ± 8.5,5,60
4,HiCARN,0.241 ± 0.08,82.96 ± 12.9,84.62 ± 6.8,77.24 ± 12.9,5,60
5,HiCDiff,0.416 ± 0.09,64.32 ± 4.1,78.08 ± 3.0,62.70 ± 1.4,5,60
6,EMMA,0.291 ± 0.03,87.79 ± 4.8,82.54 ± 1.3,84.34 ± 5.3,5,60


In [6]:
TABLE_IV = compute_ablation_table()

In [7]:
display(Markdown("## Table IV. EMMA ablation"))
display(TABLE_IV)

## Table IV. EMMA ablation

,Variant,HiCRep ↑,GDISCO ↑,DI Range ↑
0,EMMA,85.51 ± 4.6,72.35 ± 8.6,73.71 ± 5.8
1,- EMD,76.72 ± 6.7,72.55 ± 7.4,69.91 ± 4.2
2,- MaskAE,19.25 ± 6.9,64.29 ± 7.4,71.14 ± 9.8
3,- MWR,85.68 ± 4.4,72.40 ± 8.5,72.76 ± 5.4
4,- DDN,53.50 ± 13.0,69.16 ± 7.9,67.89 ± 3.7
